In [0]:
import json
import os 
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL ='https://cvijan.github.io/nss-api/files/'
RAW_DIR ='data/raw'
PROCESSED_DIR ='data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok= True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
# 1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев. 

def get_json(url: str) -> dict:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()

In [0]:
# 2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next
# линкове и враћа записе са свих страна у једној листи

def fetch_all(first_url: str) -> list:
    records = [] 
    url = first_url
    while url:
        page = get_json(url)
        records.extend(page['data'])
        url = urljoin(url, page['next']) if page['next'] else None
    return records

In [0]:
# 3. Чување сирових података. Написати функцију save_json(records: list, path: str) -> None која листу речника уписује у JSON фајл без икаквих измена.

def save_json(records: list, path: str) -> None:
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

save_json(orders, os.path.join(RAW_DIR, 'orders.json'))

In [0]:
# 4. Преузимање фајла.
def download_file(url: str, path: str) -> int:
    """Download a file and save it unchanged. Returns its size in bytes."""
    response = requests.get(url, timeout=30)
    response.raise_for_status() # 404 and other errors stop here
    with open(path, 'wb') as f: # 'wb' = write bytes, needed for Parquet
        f.write(response.content) # .content = the file's bytes, unchanged
    return len(response.content)

In [0]:
# 5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у
# raw_dir и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams',
# 'matches'

def extract(raw_dir: str) -> dict :
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    save_json(orders, os.path.join(raw_dir, 'orders.json'))
    download_file(FILES_URL + 'products.csv', os.path.join(raw_dir, 'products.csv'))
    download_file(FILES_URL + 'teams.csv', os.path.join(raw_dir, 'teams.csv'))
    download_file(FILES_URL + 'matches.parquet', os.path.join(raw_dir, 'matches.parquet'))
    df_orders = pd.json_normalize(orders, sep = '_')
    df_orders = df_orders.rename(columns = {
        'customer_name': 'customer',
        'customer_city': 'city'
    })
    df_products = pd.read_csv(os.path.join(raw_dir, 'products.csv'))
    df_teams = pd.read_csv(os.path.join(raw_dir, 'teams.csv'))
    df_matches = pd.read_parquet(os.path.join(raw_dir, 'matches.parquet'))

    return {
        'orders': df_orders,
        'products': df_products,
        'teams': df_teams,
        'matches': df_matches
    }

In [0]:
# 6. Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame
# која чисти каталог производа

def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    duplikati = df.duplicated(subset=['product_id'])
    df = df.loc[~duplikati]
    df['team_code'] = df['team_code'].fillna('').str.strip().str.upper()
    df['category'] = df['category'].str.strip().str.lower()
    df['price'] = df['price'].str.replace(' RSD', '', regex=False)
    df['price'] = df['price'].str.replace(',', '.', regex=False)
    df['price'] = pd.to_numeric(df['price'])
    df['active'] = df['active'].str.strip().str.lower() == 'yes'

    return df

In [0]:
# 7. Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame
# order_id, customer, city, payment без размака;
# order_id велика слова
# customer и city -> title()
# Град који недостаје (isna) -> 'Unknown'

def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in ['order_id', 'customer', 'city', 'payment']:
        df[col] = df[col].str.strip()
    df['order_id'] = df['order_id'].str.upper()
    df['customer'] = df['customer'].str.title()
    df['city'] = df['city'].str.title().fillna('Unknown')
    return df

In [0]:
# 8. Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple
# која враћа (unique, duplicates) по order_id, задржава први

def remove_duplicates(df: pd.DataFrame) -> tuple:
    is_dup = df.duplicated('order_id', keep='first')
    return df[~is_dup], df[is_dup]

In [0]:
# 9. Написати функцију validate_order(row: dict, product_ids: set) -> str која враћа
# 'OK' за исправан ред, у супротном разлог
# Провере радити редом и вратити први пронађени проблем
# order_id не почиње са NSS- -> 'invalid_order'
# quantity недостаје -> 'missing_quantity'
# quantity није цео број већи од нуле -> 'invalid_quantity'
# unit_price није број већи од нуле -> 'invalid_price'
# product_id не постоји међу производима -> 'unknown_product'
# Помоћ:
# pd.to_numeric(value, errors='coerce') претвара '2' у 2, а 'abc' у NaN
# pd.isna(value) препознаје и None и NaN

def validate_order(row: dict, product_ids: set) -> str:
    quantity = pd.to_numeric(row['quantity'], errors='coerce')
    price = pd.to_numeric(row['unit_price'], errors='coerce')
    if not row['order_id'].startswith('NSS-'):
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    if pd.isna(quantity) or quantity <= 0 or quantity != int(quantity):
        return 'invalid_quantity'
    if pd.isna(price) or price <= 0:
        return 'invalid_price'
    if row['product_id'] not in product_ids:
        return 'unknown_product'
    return 'OK'
    

In [0]:
# 10. Написати функцију transform_orders(df: pd.DataFrame, products: pd.DataFrame) ->
# pd.DataFrame

# Типови:
# quantity -> int
# unit_price -> float
# order_date -> датум (pd.to_datetime)
# Нова колона total (quantity * unit_price)
# Из очишћених производа (products) додати product_name, team_code и category
# (merge по product_id)
# Output: 1800 редова и 12 колона (8 из поруџбина, total, и 3 из производа)

def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['quantity'] = pd.to_numeric(df['quantity']).astype(int)
    df['unit_price'] = pd.to_numeric(df['unit_price']).astype(float)
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['total'] = df['quantity'] * df['unit_price']
    names = products[['product_id', 'product_name', 'team_code', 'category']].rename(columns={'name': 'product_name'})
    return df.merge(names, on='product_id', how='left')


In [0]:
# 11. Написати функцију split_orders(df: pd.DataFrame, products: pd.DataFrame) ->
# tuple која сирове поруџбине чисти, избацује дупликате, проверава и враћа торку (valid,
# rejected)
# valid су трансформисани исправни редови
# rejected су очишћени неисправни редови и дупликати, са колоном reason
# (дупликати за разлог треба да имају 'duplicate')
# Output: 1800 исправних, 52 одбијених (од тога 12 дупликата)

def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    df = clean_orders(df)
    unique, duplicates = remove_duplicates(df)
    duplicates = duplicates.assign(reason='duplicate')
    product_ids = set(products['product_id'])
    reasons = unique.apply(lambda row: validate_order(row, product_ids), axis=1)
    valid = transform_orders(unique[reasons == 'OK'], products)
    rejected = pd.concat([unique[reasons != 'OK'].assign(reason=reasons[reasons != 'OK']), duplicates])
    return valid, rejected
# products = clean_products(data['products'])
# valid, rejected = split_orders(data['orders'], products)
# print (rejected.shape)

In [0]:
# 12. Резултати тимова. Написати функцију team_results(matches: pd.DataFrame, teams:
# pd.DataFrame) -> pd.DataFrame која сваку одиграну утакмицу претвара у два реда, један
# за домаћина и један за госта.
# Користити само утакмице са status == 'finished' (одложена нема резултат)
# Колоне:
# match_id
# match_date (Датум из kickoff_ts, без времена -
# df['kickoff_ts'].dt.normalize())
# team_code, opponent_code
# goals_for, goals_against
# result ('W', 'D', 'L'), points (3, 1, 0)
# Утакмице имају team_id, а производи team_code

def team_results(matches: pd.DataFrame, teamspd.DataFrame) -> pd.DataFrame:
    



In [0]:
# 17.

def load(outputs: dict, processed_dir: str) -> dict:
    written = {}
    for name, df in outputs.items():
        path = os.path.join(processed_dir, name)
        if name.endswith('.parquet'):
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)
        written[name] = len(df)
    return written

In [0]:
# 18.

def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    # results = team_results(data['matches'], data['teams'])
    # windows = match_windows(results)
    # outputs = {
    #     'clean_orders.parquet': valid,
    #     'rejected_orders.csv': rejected,
    #     'products.csv': products,
    #     'league_table.csv': league_table(results),
    #     'revenue_by_team.csv': revenue_by(valid.dropna(subset=['team_code']), 'team_code'),
    #     'revenue_by_category.csv': revenue_by(valid, 'category'),
    #     'sales_after_results.csv': sales_after_results(valid, windows),
    # }
    # written = load(outputs, processed_dir)
    # summary = {
    #     'total_rows': len(data['orders']),
    #     'duplicates': int((rejected['reason'] == 'duplicate').sum()),
    #     'valid_rows': len(valid),
    #     'rejected_rows': len(rejected),
    #     'total_revenue': round(float(valid['total'].sum()), 2),
    #     'files_written': len(written) + 1,
    # }
    # pd.DataFrame(list(summary.items()), columns=['metric',
    # 'value']).to_csv(os.path.join(processed_dir, 'summary.csv'), index=False)
    # return summary